# D1.3 · Agent-assisted detection engineering

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *AI for Security*

---

**Risk.** Coverage gaps nobody mapped.

**Control.** Detection-as-code with agents inside the CI loop.

**This lab.** Generate, unit-test and tune detections inside CI.

| | |
|---|---|
| Open-source tooling | Sigma, Wazuh |
| Open-weight models | Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("D1.3"))

Agent-assisted detection engineering: the agent writes candidate rules, and you keep the thing that decides whether they are any good.

In [ ]:
from cybercommons import soc
import time

# three candidate rules an agent might propose for the same concern
now = time.time()
events  = [soc.Event(now + i, "patch-agent", "http_get", "https://api.github.com/x")
           for i in range(20)]
events += [soc.Event(now + 21, "patch-agent", "http_get",
                     "http://169.254.169.254/latest/meta-data/")]

candidates = {
 "any http_get":        soc.Rule("any http_get", "low",
                                 lambda e: e.action == "http_get", "review"),
 "non-allowlisted host": soc.Rule("non-allowlisted host", "high",
                                  lambda e: "api.github.com" not in (e.target or ""),
                                  "block egress and rotate"),
 "metadata service":     soc.Rule("metadata service", "critical",
                                  lambda e: "169.254.169.254" in (e.target or ""),
                                  "kill session, rotate instance role"),
}
truth = {"patch-agent:http_get"}     # only the metadata call is genuinely bad
for name, rule in candidates.items():
    alerts = soc.run_rules(events, [rule])
    q = soc.triage_quality(alerts, truth)
    print(f"{name:22s} alerts={q['alerts']:3d} precision={q['precision']:.3f} "
          f"per-TP={q['alerts_per_true_positive']}")

All three rules 'work'. Only one is deployable. The engineering judgment the agent cannot supply is the cost of the false positives — because that cost is measured in analyst trust, which is not in the telemetry.

### Expect

The broad rule fires 21 times, the host rule once, the metadata rule once — with precision rising and alerts-per-true-positive falling across the three.

### Your turn

Have an agent generate five rules for one concern in your environment, then score them against a week of real telemetry before deploying any. The scoring step is the job.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/D1.3.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*